In [ ]:
import torch
import torchvision.transforms as transforms
from tqdm import tqdm
import requests
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig, create_reference_model
from accelerate import init_empty_weights
from transformers import BitsAndBytesConfig
from trl.models import AutoModelForCausalLMWithValueHead
import os
import torch




import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from accelerate import init_empty_weights
from transformers import BitsAndBytesConfig
import os
from datasets import load_dataset

# Load the dataset
val = pd.read_excel("/home/ml/--------/ollamawork/Finetuning_2025/Final_training_dataset_bhusan.xlsx")
#val = pd.read_excel(val)
# Define the model name
test_dataset = val.iloc[:10, :]

col = {'Test Case Name' : 'name', 'Steps with Expected Output': 'steps', 'Python Code': 'python code', 'Summary': 'summary'}
test_dataset.rename(columns = col, inplace=True)

test_dataset.to_csv('test1.csv', index = True)


data_files = {"test": "test1.csv"}
dataset = load_dataset("csv", data_files=data_files)
model_name = '/home/ml/--------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.2_1'

# Load the tokenizer and model with 4-bit precision
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=False
)


# Load the fine-tuned model

#base_model1 = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, torch_dtype=torch.bfloat16,  device_map="cuda",trust_remote_code=False,low_cpu_mem_usage=True)
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name,quantization_config=bnb_config, torch_dtype=torch.bfloat16, device_map="cuda", trust_remote_code=False, low_cpu_mem_usage=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
# Set the padding token to eos_token
tokenizer.truncation_side = "right"
tokenizer.pad_token = tokenizer.eos_token


# Prepare dataset for PPO training


        
def preprocess_data(row):  # Ensure left padding before tokenizing
    batch_encoded = []
    labels = []
    """
<|im_start|>system<|im_sep|>
Generate a complete Python code based on the following test case summary and test steps. Think step by step while generating the output. Put non-python text in comments.<|im_end|>
<|im_start|>user<|im_sep|>\n
Name: {row['name']}\n
Test Case: {row['summary']}\n
Test Steps: {row['steps']}\n
Generated Code: 
        """
    query = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Generate a complete Python code based on the following test case summary and test steps. Think step by step while generating the output. Put non-python text in comments.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Name: {row['name']}\n
Test Case: {row['summary']}\n
Test Steps: {row['steps']}\n
Python Code:
        """
    encoded = tokenizer(
        query,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=600
    )

    input_ids = encoded["input_ids"].tolist()
    batch_encoded.append(input_ids)
    labels.append(input_ids)

    # Apply consistent left-padding
    padded_batch = tokenizer.pad({"input_ids": batch_encoded, "labels": labels}, padding=True, return_tensors="pt")
    
    return padded_batch


columns_to_remove = ["name", "steps", "python code", "summary"]
tokenized_dataset_train = dataset['test'].map(
    preprocess_data,
    batched=False,
    remove_columns=columns_to_remove,
    load_from_cache_file=False
)
tokenized_dataset_train.set_format(type='torch')


In [2]:
#ref_model = create_reference_model(base_model)

In [ ]:
ppo_config = PPOConfig(
    model_name=base_model,
    learning_rate=5e-5, 
    batch_size=1,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    early_stopping=True,
    target_kl=0.001,  # Keep PPO close to base model
    kl_penalty="kl",
    init_kl_coef=0.01,
    adap_kl_ctrl=True,
    whiten_rewards=True,  # Normalize rewards for stability
    max_grad_norm=0.5 , # Adaptive KL control
    cliprange_value=0.01
)

def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])
# Create reference model for PPO
ppo_trainer = PPOTrainer(
    model=base_model,
    ref_model=base_model,  # Use the same base model as the reference model
    config=ppo_config,
    tokenizer=tokenizer,
    dataset=tokenized_dataset_train,
    data_collator=collator
)



import re
def get_reward_score(prompt):

    # Define the URL for the Ollama API (replace with your actual API URL if different)
    url = "http://localhost:11434/api/generate"  # The URL for the Ollama Phi4 API
    model = "phi4_modelfile_RL:latest"  # The model we're usin
    prompt1=f"""
<|user|>
### The text contains test case description, test steps, test code for that test case generated by finetuned LLM and the actual test code. Your task is to evaluate the test code by comparing it with the actual code and give a score between 0 and 1. The output should contain only one number between 0 and 1 based on your evaluation.

### Evaluation Guidelines:  
- **Score:** the generated code on a scale of 0 to 1.

### Scoring Guide:  
- **0.1-0.3:** Very poor results. The code has major issues or hallucinations and requires significant modifications to run the test case.  
- **0.4-0.5:** Unusable code. It is somewhat related to the input steps but misses key objectives. Debugging would take more time than rewriting.  
- **0.6-0.7:** Adequate results. The code is usable but needs adjustments, such as fixing incomplete functions or incorrect function calls.  
- **0.8:** Good results. The code achieves the objective, with only minor issues (e.g., logging errors, variable name mismatches).  
- **0.9-1:** Very good results. The code aligns well with the steps, achieves the objective, and can be executed without issues.  


Ensure your evaluation includes only one number between 0 and 1.
{prompt}

"""
    data = {
        "model": model,
        "prompt": prompt1,
        "stream": False
    }

    headers = {
        "Content-Type": "application/json",
    }
    # Send POST request to the Ollama API
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        response_json = response.json()
        response_text = response_json.get("response", "").strip()
        #print(response_text)

        # Try to convert the response to a float
        try:
            return float(response_text)
        except ValueError:
            # If conversion fails, look for the Score: in the output and extract the number
            match = re.search(r"Score:\s*(\d\.\d+)", response_text)
            if match:
                return float(match.group(1))
            else:
                return 0.0  # Default to 0 if no valid score is found

    except requests.exceptions.RequestException as e:
        print(f"Error occurred: {e}")
        return 0.0
# Track training stats
training_stats = []

# Training Loop

#print(len(tokenized_dataset_train["input_ids"]))



In [ ]:
import re
num_epochs=2

rewrads_epoch=[]
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    n=0
    rewards_batch=[]
    for batch_idx, batch in enumerate(tqdm(ppo_trainer.dataloader)):

#for epoch, batch in enumerate(tqdm(ppo_trainer.dataloader)):
        query_tensors = [torch.tensor(q).to('cuda') for q in batch["input_ids"]]
        
        #print(query_tensors)
        #print("Length: ", len(query_tensors))
        #print(query_tensors)
        response_tensors = []
        query = query_tensors[0]
        #print(query)
        query = query.view(-1, query.shape[-1]).to("cuda")
        #print('Q2',query)
        #print('Q3',query_x)
        '''response = ppo_trainer.generate(query[0], max_new_tokens= 300,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id, temperature=0.1)'''
        #response_text = tokenizer.decode(response[0], skip_special_tokens=True)
        '''print(len(response[0]))
        # Print outputs
        print("PPO Response (Tensor):", response)
        print("Decoded Response (Text):", response_text)
        response_len = len(response[0]) - len(query[0]) 
        #print("PPO Only code token : ", response[0][-response_len:])
        code_text = "".join(tokenizer.batch_decode(response[0][-response_len:], skip_special_tokens=True))'''
        #print("PPO Only code: ", code_text)
        '''query_text = tokenizer.decode(query.squeeze(0).tolist(), skip_special_tokens=True)

        print("Decoded Query Text:", query_text)
        input=tokenizer(query_text,
            return_tensors="pt",
            return_attention_mask=True,  # Enable attention mask
            padding=True,
            truncation=True)

        # Step 2: Generate a response from the base model
        #query_tensor = tokenizer(query_text, return_tensors="pt").input_ids.to('cuda')
        input = {key: value.to('cuda') for key, value in input.items()}
        response_tokens = base_model.generate(**input, max_new_tokens=512)

        # Step 3: Decode the generated response to text
        #response_text = tokenizer.decode(response_tensor[0], skip_special_tokens=True)
        print("Tokenized Response:", response_tokens)
        print(len(query.squeeze(0)))
        print(len(response_tokens))

        # Step 4: Tokenize the generated response
        response_text = tokenizer.batch_decode(response_tokens, skip_special_tokens=True)[0]
        print("Generated Response:", response_text)
        response_len = len(response_tokens[0]) - len(query.squeeze(0).tolist()) 
        print("Only code token : ", response_tokens[0][-response_len:])
        code_text = "".join(tokenizer.batch_decode(response_tokens[0][-response_len:], skip_special_tokens=True))
        print("Only code: ", code_text)'''

        # Generate response
        index=[]
    
        for query in query_tensors:
            #print(f"Query tensor shape: {query.shape}")
            query = query.view(-1, query.shape[-1]).to("cuda")  # Ensure correct shape
        #  output = base_model.generate(query[0], max_new_tokens=50)
        #  print("Generated output:", tokenizer.decode(output[0], skip_special_tokens=True, do_sample=False))

            #print(f"Query tensor shape: {query.shape}")
            #print(query)
            #query = query.squeeze(0) 
            #print('q', query)
            '''query_text = tokenizer.decode(query.tolist(), skip_special_tokens=True)
            input=tokenizer(query_text, return_tensors="pt", return_attention_mask=True, padding=True, truncation=True)'''

        # Step 2: Generate a response from the base model
        #query_tensor = tokenizer(query_text, return_tensors="pt").input_ids.to('cuda')
            #input = {key: value.to('cuda') for key, value in input.items()}
            #query_response = base_model.generate(**input, max_new_tokens=256,do_sample=True,max_length=512, pad_token_id=tokenizer.eos_token_id,temperature=0.1)
            torch.manual_seed(42)
            query_response=ppo_trainer.generate(query[0], max_new_tokens= 3600,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id, temperature=0.1, repetition_penalty=1.1)
            #response_text = tokenizer.decode(response[0], skip_special_tokens=False)
            #print(len(query[0]))
            print((query_response[0]))
            #query_response = ppo_trainer.generate(query, **generation_kwargs)
            response_len = len(query_response[0]) - len(query[0]) 
            #print((response_len))
            #print("Query response: ", query_response[0][-response_len:])
            response_tensors.append(query_response[0][-response_len:])
            index.append(n)
            n=n+1
            if n%1==0:
               print(n)
            
            #response_tensors.append(query_response)
        #batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]
        #batch["response"]  = ["".join(tokenizer.batch_decode(r, skip_special_tokens=True)) for r in response_tensors]
        if batch is None:
            batch = {}  # Ensure batch is a dictionary

        if response_tensors is None:
            response_tensors = []

        batch["response"] = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
        #batch["response"] = [tokenizer.batch_decode(r, skip_special_tokens=True)[0] for r in response_tensors]
        #print(response_tensors)
        #print(batch["response"])

        # Compute rewards
        texts = [tokenizer.decode(q.squeeze().tolist(), skip_special_tokens=True) + r for q, r in zip(batch["input_ids"], batch["response"])]
        #pipe_outputs = get_reward_score(texts)
        #print(texts)
        #print(index)
        '''texts = [
    "Name:" + text.split("Name:", 1)[-1] + '\n' + 
    'Actual Code' + '\n' + (val['Python Code'][val['Test Case Name'].str.strip().eq(text.split("Name", 1)[-1].strip().split("\n", 1)[0].strip())].values[0]
    if text.split("Name:", 1)[-1].strip().split("\n", 1)[0].strip() in val['Test Case Name'].str.strip().values else "No Matching Test Code")
    + '\n' +
    "\nCompare the actual code and the generated code and return only a single float number between 0 and 1 based on your evaluation.\n<|end|>\n<|assistant|>\n"
    for text, n in zip(texts, index)]'''
        texts = [
    "Name:" + text.split("Name:", 1)[-1] + '\n' + 
    '###Actual Code:' + '\n' + 
    (val.loc[val['Test Case Name'].str.strip().eq(text.split("Name:", 1)[-1].strip().split("\n", 1)[0].strip()), 'Python Code'].iloc[0]
     if not val.loc[val['Test Case Name'].str.strip().eq(text.split("Name:", 1)[-1].strip().split("\n", 1)[0].strip()), 'Python Code'].empty 
     else "No Matching Test Code") + '\n' +
    "\nCompare the actual code and the generated code and return only a single float number between 0 and 1 based on your evaluation.\n<|end|>\n<|assistant|>\n"
    for text, n in zip(texts, index)
]


        #print(texts)
        for text in texts:
            print("Test;",text)
            print("\n" + "-" * 80 + "\n")


        
        #rewards = [[torch.tensor(score)] for score in pipe_outputs]
        rewards = [torch.tensor(get_reward_score(text)) for text in texts]
        #print(rewards)
        rewards_batch.append(rewards)

        # Logging detailed info
        print(rewards)

        




        #print(f"Epoch {epoch+1}: Reward = {rewards}")  # Print live updates

        # PPO Step
        #query_tensors = [torch.tensor(q).view(-1, len(q)).to('cuda') for q in tokenized_dataset_train["input_ids"]]
        #query_tensors = [torch.tensor(q, dtype=torch.long).unsqueeze(0).to('cuda') for q in tokenized_dataset_train["input_ids"]]
        query_tensors=[tensor.squeeze(0) for tensor in query_tensors]
        query_tensors=[tensor.squeeze(0) for tensor in query_tensors]
        #response_tensors= [tensor.unsqueeze(0) for tensor in response_tensors]
        #print('Rewards:  ', rewards)
        rewards= [tensor.unsqueeze(0) for tensor in rewards]
        #print(rewards)
        eos_token_id = tokenizer.eos_token_id  # Replace with your actual EOS token ID
        max_length = 256


        # Pad each tensor to the max_length
        '''padded_tensors = [
            torch.nn.functional.pad(tensor, (0, max_length - tensor.shape[0]), value=eos_token_id)
            for tensor in query_tensors
        ]
        #print('pts: ',padded_tensors)

        #print('Q: ', query_tensors)
        #print('R: ', response_tensors)
        #print('Re: ', rewards)

        query_tensors = padded_tensors'''

        '''padded_response = [
            torch.nn.functional.pad(tensor, (0, max_length - tensor.shape[0]), value=eos_token_id)
            for tensor in response_tensors
        ]
        #print(padded_response)
        response_tensors= padded_response'''
        
        '''print("Query tensors shape:", [qt.shape for qt in query_tensors])
        print("Response tensors shape:", [rt.shape for rt in response_tensors])
        print("Rewards shape:", [reward.shape for reward in rewards])'''



        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)

        # Store training data
        training_stats.append({"epoch": epoch,  "reward": rewards})
    rewrads_epoch.append(rewards_batch[0])
# Save training logs to CSV
#print(rewrads_epoch)
import pandas as pd
df = pd.DataFrame(training_stats)
df.to_csv("training_logs.csv", index=False)

#### Save model
ppo_trainer.model.save_pretrained("./my_ppo_model3")
tokenizer.save_pretrained("./my_ppo_model3")
print("Training complete! Logs saved in 'ppo_training.log' and 'training_logs.csv'.")

import torch
import matplotlib.pyplot as plt
#flattened_list = [sum(sublist, []) for sublist in zip(*rewrads_epoch)]
# Given nested list of tensors
tensor_lists =rewrads_epoch
#print(flattened_list)
# Convert tensors to simple float lists
simple_lists = [[t.item() for t in sublist] for sublist in tensor_lists]

# Compute the average for each list
averages = [sum(lst) / len(lst) for lst in simple_lists]

# Plot the lists
print(averages)
plt.figure(figsize=(10, 5))
for i, lst in enumerate(simple_lists):
    plt.plot(lst, marker='o', label=f"List {i+1}")  # Plot list
    plt.axhline(y=averages[i], linestyle='--', label=f"Avg List {i+1}")  # Plot average line

plt.legend()
plt.xlabel("Index")
plt.ylabel("Values")
plt.title("Plot of Lists with Averages")
plt.show()

In [ ]:
import torch
import requests
import re
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead
model=AutoModelForCausalLMWithValueHead.from_pretrained('/home/ml/Bhushan - Thales/Finetunning/my_ppo_model4', torch_dtype=torch.bfloat16, device_map="cuda")
ref_model=AutoModelForCausalLM.from_pretrained('/home/ml/Bhushan - Thales/Finetunning/Result_phi2_4', torch_dtype=torch.bfloat16, device_map="cuda")
import re

  # Return an empty string if triple quotes are not found

    



def get_reward_score(prompt):

    # Define the URL for the Ollama API (replace with your actual API URL if different)
    url = "http://localhost:11434/api/generate"  # The URL for the Ollama Phi4 API
    model = "phi4_modelfile_RL"  # The model we're usin
    prompt1=f"""
<|user|>
### The text contains test cases, description, input, output, test code for the test case generated by finetuned LLM and the actual test code. Your task is to evaluate the test code by comparing it with the actual code and give a score between 0 and 1. The output should contain only one number between 0 and 1 based on your evaluation.

### Evaluation Guidelines:  
- **Score:** the generated code on a scale of 0 to 1.

### Scoring Guide:  
- **0.1-0.3:** Very poor results. The code has major issues or hallucinations and requires significant modifications to run the test case.  
- **0.4-0.5:** Unusable code. It is somewhat related to the input steps but misses key objectives. Debugging would take more time than rewriting.  
- **0.6-0.7:** Adequate results. The code is usable but needs adjustments, such as fixing incomplete functions or incorrect function calls.  
- **0.8:** Good results. The code achieves the objective, with only minor issues (e.g., logging errors, variable name mismatches).  
- **0.9-1:** Very good results. The code aligns well with the steps, achieves the objective, and can be executed without issues.  


Ensure your evaluation includes only one number between 0 and 1.
{prompt}
###Actual code:
"""
    data = {
        "model": model,
        "prompt": prompt1,
        "stream": False
    }

    headers = {
        "Content-Type": "application/json",
    }
    # Send POST request to the Ollama API
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        response_json = response.json()
        response_text = response_json.get("response", "").strip()
        #print(response_text)

        # Try to convert the response to a float
        try:
            return float(response_text)
        except ValueError:
            # If conversion fails, look for the Score: in the output and extract the number
            match = re.search(r"Score:\s*(\d\.\d+)", response_text)
            if match:
                return float(match.group(1))
            else:
                return 0.0  # Default to 0 if no valid score is found

    except requests.exceptions.RequestException as e:
        print(f"Error occurred: {e}")
        return 0.0
#print(filt_s(get_between_triple_quotes(dataset['test']['answer'])))
tokenizer = AutoTokenizer.from_pretrained('/home/ml/Bhushan - Thales/Finetunning/my_ppo_model4')
def preprocess_data(examples):
    batch_encoded = []
    labels = []
    for i, (name, steps, answer, summary) in enumerate(zip(examples['name'], examples["steps"], examples["python code"], examples["summary"])):
        prompt = f"""
<|im_start|>system<|im_sep|>
Generate a complete Python code based on the following test case summary and test steps. Think step by step while generating the output. Put non-python text in comments.<|im_end|>
<|im_start|>user<|im_sep|>
Name: {name}\n
Test Case: {summary}\n
Test Steps: {steps}\n
Python Code: {answer}<|im_end|>
<|im_start|>assistant<|im_sep|>
        """
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=256
    )

    input_ids = encoded["input_ids"].tolist()
    batch_encoded.append(input_ids)
    labels.append(input_ids)

    # Apply consistent left-padding
    padded_batch = tokenizer.pad({"input_ids": batch_encoded, "labels": labels}, padding="longest", return_tensors="pt")
    
    return padded_batch

# Load the tokenizer


#tokenized_dataset_train.set_format(type='torch')
# Load the dataset
val = pd.read_excel("/home/ml/--------/ollamawork/Finetuning_2025/Final_training_dataset_bhusan.xlsx")
val1 = pd.read_excel("/home/ml/--------/ollamawork/Finetuning_2025/Final_training_dataset_bhusan.xlsx")

# Preprocess and rename columns
col = {'Test Case Name' : 'name', 'Steps with Expected Output': 'steps', 'Python Code': 'python code', 'Summary': 'summary'}
val.rename(columns=col, inplace=True)

# Save to CSV
val.to_csv('test1.csv', index=True)

# Load dataset from CSV
data_files = {"test": "test1.csv"}
dataset = load_dataset("csv", data_files=data_files)

# Set batch size and format dataset
batch_size = len(val)
dataset.set_format("pandas")

# Sample a batch of data
df_batch = dataset['test'].to_pandas()

# Preprocess each row in the batch
'''batch_data = []
for idx, row in df_batch.iterrows():
    batch_data.append(preprocess_data(row, tokenizer))'''
columns_to_remove=["name", "steps", "python code", "summary"]
tokenized_dataset_train = dataset['test'].map(
    preprocess_data,
    batched=False,
    remove_columns=columns_to_remove
)
print(tokenized_dataset_train)
#print(list(q for q in tokenized_dataset_train["input_ids"]))

#print(filt_s(get_between_triple_quotes(dataset['test']['answer'][0])))
# Now batch_data contains padded inputs and labels for the batch


# Initialize response lists
response_tensors_ref, response_tensors = [], []
device='cuda'
# Get response from the reference and target models
# Convert each element to a proper list format, then create tensors
query_tensors = [torch.tensor(list(q), dtype=torch.long).to('cuda') for q in tokenized_dataset_train["input_ids"]]

for i in range(batch_size):
    query = torch.tensor(query_tensors[i]).to(device)

    gen_len = 512  # Assuming you have a function for this
    query_response_ref = ref_model.generate(query[0], max_new_tokens= 512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id, temperature=0.1)

    # Store reference model's response
    response_len_ref = len(query_response_ref) - len(query)
    response_tensors_ref.append(query_response_ref[-response_len_ref:])

    # Get target model's response
    query_response = model.generate(query[0], max_new_tokens= 512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id, temperature=0.1)

    # Store target model's response
    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])

# Decode responses
game_data = {"query": df_batch["question"].tolist()}
game_data["response (before)"] = [tokenizer.decode(list(response_tensors_ref[i][0]), skip_special_tokens=True) for i in range(batch_size)]

game_data["response (after)"] = [tokenizer.decode(list(response_tensors[i][0]), skip_special_tokens=True) for i in range(batch_size)]

# Sentiment analysis of query/response pairs before and after
texts_before = [q + r for q, r in zip(game_data["query"], game_data["response (before)"])]
texts_before=[
    "###Endpoint:" + text.split("###Endpoint:", 1)[-1] + '\n' + 
    'Actual Code' + '\n' + (val1['Test_Code'][val1['Description'].str.strip().eq(text.split("###Description:", 1)[-1].strip().split("\n", 1)[0].strip())].values[0]
    if text.split("###Description:", 1)[-1].strip().split("\n", 1)[0].strip() in val1['Description'].str.strip().values else "No Matching Test Code")
    + '\n' +
    "\nCompare the actual code and the generated code and return only a single float number between 0 and 1 based on your evaluation.\n<|end|>\n<|assistant|>\n"
for text in texts_before]
pipe_outputs_before = [(get_reward_score(text)) for text in texts_before]
print(pipe_outputs_before)
positive_scores_before =pipe_outputs_before

game_data["rewards (before)"] = positive_scores_before

texts_after = [q + r for q, r in zip(game_data["query"], game_data["response (after)"])]
texts_after=[
    "###Endpoint:" + text.split("###Endpoint:", 1)[-1] + '\n' + 
    'Actual Code' + '\n' + (val1['Test_Code'][val1['Description'].str.strip().eq(text.split("###Description:", 1)[-1].strip().split("\n", 1)[0].strip())].values[0]
    if text.split("###Description:", 1)[-1].strip().split("\n", 1)[0].strip() in val1['Description'].str.strip().values else "No Matching Test Code")
    + '\n' +
    "\nCompare the actual code and the generated code and return only a single float number between 0 and 1 based on your evaluation.\n<|end|>\n<|assistant|>\n"
for text in texts_after]
pipe_outputs_after = [(get_reward_score(text)) for text in texts_after]

positive_scores_after = pipe_outputs_after

game_data["rewards (after)"] = positive_scores_after
for key, value in game_data.items():
    print(f"{key}: {len(value)}")

# Store results in a dataframe
df_results = pd.DataFrame(game_data)
df_results